In [ ]:
pip install dask[complete] numpy pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Membaca dataset
import pandas as pd
df = pd.read_csv("Dataset_Ojol_Transaction.csv")
print(df.head())
print(df.info())
print("Dataset shape:", df.shape)

     id                                   date  mode  \
0  1617  2019/03/09 20:45 s/d 2019/03/09 19:55  BIKE   
1  1297  2019/03/09 19:55 s/d 2019/03/09 19:54  FOOD   
2  1394  2019/03/09 19:54 s/d 2019/03/09 18:56  SHOP   
3  1120  2019/03/09 18:56 s/d 2019/03/09 12:28  FOOD   
4  2053  2019/03/09 12:28 s/d 2019/03/08 18:25   CAR   

                                         from_alamat   from_kelurahan  \
0                    Gang Ikhwan No.16,  Sungai Jawi      DARAT SEKIP   
1  Neo Shabu-Shabu Steak & Shake, Johar, Jl. Joha...  SUNGAI BANGKONG   
2             Alfamart Pontianak Mall, Jl Teuku Umar      DARAT SEKIP   
3  Parklife, Jl. Karimata No.64, Sungai Bangkong,...          MARIANA   
4        Jl. Tabrani Ahmad No.12,  Sungai Jawi Dalam         PAL LIMA   

    from_kecamatan               from_latlng  \
0   PONTIANAK KOTA     -0,0303277,109,297753   
1   PONTIANAK KOTA       -0,02861,109,329253   
2   PONTIANAK KOTA    -0,0301863,109,3356331   
3   PONTIANAK KOTA    -0,0305815

In [ ]:
# EDA
print("\nDeskripsi Statistik:")
print(df.describe(include='all'))


Deskripsi Statistik:
                 id                                   date  mode  \
count   1017.000000                                   1017  1017   
unique          NaN                                   1017     4   
top             NaN  2018/09/10 09:12 s/d 1900/01/00 00:00  BIKE   
freq            NaN                                      1   325   
mean    1621.648968                                    NaN   NaN   
std      296.954571                                    NaN   NaN   
min     1101.000000                                    NaN   NaN   
25%     1367.000000                                    NaN   NaN   
50%     1624.000000                                    NaN   NaN   
75%     1878.000000                                    NaN   NaN   
max     2132.000000                                    NaN   NaN   

                  from_alamat from_kelurahan  from_kecamatan  \
count                    1017           1017            1017   
unique                    504    

In [ ]:
# Cek missing values
print("\nJumlah Missing Values Tiap Kolom:")
print(df.isnull().sum())


Jumlah Missing Values Tiap Kolom:
id                            0
date                          0
mode                          0
from_alamat                   0
from_kelurahan                0
from_kecamatan                0
from_latlng                   0
to_alamat                     3
to_kelurahan                  0
to_kecamatan                  0
to_latlng                     0
distance                      0
amount_delivery               0
amount_merchant               0
transaction_amount_total      0
customer_id                   0
customer_gender               0
customer_birthdate            0
driver_id                     0
driver_gender                 0
driver_birthdate              0
kendaraan_jenis               0
kendaraan_merk                0
merchant_id                 567
merchant_name               567
merchant_category           567
dtype: int64


In [ ]:
# Handling Missing Values
# Identifikasi Kolom Numerik & Kategorikal
num_cols = df.select_dtypes(include=["number"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

# Cek missing values per kolom
missing = df.isnull().sum()
print("\nJumlah missing values per kolom:\n", missing)

# Isi missing value numerik dengan mean per kolom
for col in num_cols:
    mean_val = df[col].mean()
    df[col] = df[col].fillna(mean_val)

# Isi missing value kategorikal dengan "Unknown"
for col in cat_cols:
    df[col] = df[col].fillna("Unknown")

# Cek ulang setelah handling
print("\nMissing values setelah di-handle:\n", df.isnull().sum())


Jumlah missing values per kolom:
 id                            0
date                          0
mode                          0
from_alamat                   0
from_kelurahan                0
from_kecamatan                0
from_latlng                   0
to_alamat                     3
to_kelurahan                  0
to_kecamatan                  0
to_latlng                     0
distance                      0
amount_delivery               0
amount_merchant               0
transaction_amount_total      0
customer_id                   0
customer_gender               0
customer_birthdate            0
driver_id                     0
driver_gender                 0
driver_birthdate              0
kendaraan_jenis               0
kendaraan_merk                0
merchant_id                 567
merchant_name               567
merchant_category           567
dtype: int64

Missing values setelah di-handle:
 id                          0
date                        0
mode                    

In [ ]:
# Encoding kolom customer_gender
if "customer_gender" in df.columns:
    df["customer_gender_encoded"] = df["customer_gender"].map({
        "l": 0,
        "p": 1,
        "x": 2,
        "unknown": -1
    })

# Encoding kolom kendaraan_jenis
if "kendaraan_jenis" in df.columns:
    df["kendaraan_jenis_encoded"] = df["kendaraan_jenis"].map({
        "motor": 0,
        "mobil": 1,
        "unknown": -1
    })

# Tampilkan hasil cek
print("\nContoh hasil encoding:")
print(df[["customer_gender", "customer_gender_encoded",
          "kendaraan_jenis", "kendaraan_jenis_encoded"]].head(10))


Contoh hasil encoding:
  customer_gender  customer_gender_encoded kendaraan_jenis  \
0               P                      NaN           MOTOR   
1               L                      NaN           MOTOR   
2               L                      NaN           MOTOR   
3               L                      NaN           MOTOR   
4               L                      NaN           MOBIL   
5               P                      NaN           MOTOR   
6               P                      NaN           MOTOR   
7               P                      NaN           MOBIL   
8               L                      NaN           MOBIL   
9               L                      NaN           MOTOR   

   kendaraan_jenis_encoded  
0                      NaN  
1                      NaN  
2                      NaN  
3                      NaN  
4                      NaN  
5                      NaN  
6                      NaN  
7                      NaN  
8                      NaN  
9  

In [ ]:
# Rapikan tanggal

# Pisahkan start_time & end_time
df[["start_time", "end_time"]] = df["date"].str.split(" s/d ", expand=True)

# Ubah ke datetime
df["start_time"] = pd.to_datetime(df["start_time"], format="%Y/%m/%d %H:%M", errors="coerce")
df["end_time"] = pd.to_datetime(df["end_time"], format="%Y/%m/%d %H:%M", errors="coerce")

# Fix kalau end_time < start_time (tukar posisi)
mask = df["end_time"] < df["start_time"]
df.loc[mask, ["start_time", "end_time"]] = df.loc[mask, ["end_time", "start_time"]].values

# Hitung durasi dalam menit
df["duration_minutes"] = (df["end_time"] - df["start_time"]).dt.total_seconds() / 60

# Tampilkan hasil
print(df[["id", "start_time", "end_time", "duration_minutes"]].head(10))

     id          start_time            end_time  duration_minutes
0  1617 2019-03-09 19:55:00 2019-03-09 20:45:00              50.0
1  1297 2019-03-09 19:54:00 2019-03-09 19:55:00               1.0
2  1394 2019-03-09 18:56:00 2019-03-09 19:54:00              58.0
3  1120 2019-03-09 12:28:00 2019-03-09 18:56:00             388.0
4  2053 2019-03-08 18:25:00 2019-03-09 12:28:00            1083.0
5  1574 2019-03-08 18:25:00 2019-03-08 18:25:00               0.0
6  1882 2019-03-08 12:58:00 2019-03-08 18:25:00             327.0
7  1958 2019-03-07 18:51:00 2019-03-08 12:58:00            1087.0
8  2009 2019-03-07 16:30:00 2019-03-07 18:51:00             141.0
9  1671 2019-03-07 15:53:00 2019-03-07 16:30:00              37.0


In [ ]:
# Feature extraction
df["year"] = df["start_time"].dt.year
df["month"] = df["start_time"].dt.month
df["day"] = df["start_time"].dt.day
df["weekday"] = df["start_time"].dt.day_name()
df["hour"] = df["start_time"].dt.hour

# Tampilkan beberapa baris data untuk dicek
print("\nHasil FE preprocessing:")
print(df[["id", "start_time", "year", "month", "day", "weekday", "hour"]].head(5))


Hasil FE preprocessing:
     id          start_time  year  month  day   weekday  hour
0  1617 2019-03-09 19:55:00  2019      3    9  Saturday    19
1  1297 2019-03-09 19:54:00  2019      3    9  Saturday    19
2  1394 2019-03-09 18:56:00  2019      3    9  Saturday    18
3  1120 2019-03-09 12:28:00  2019      3    9  Saturday    12
4  2053 2019-03-08 18:25:00  2019      3    8    Friday    18


In [ ]:
# Lowercase string
# Ambil semua kolom bertipe object (string/kategorikal)
string_cols = df.select_dtypes(include=["object"]).columns

# Ubah semua kolom string ke lowercase + strip
for col in string_cols:
    df[col] = df[col].str.lower().str.strip()

# Tampilkan beberapa baris data untuk dicek
print("\nContoh data setelah preprocessing:")
print(df.head(5))


Contoh data setelah preprocessing:
     id                                   date  mode  \
0  1617  2019/03/09 20:45 s/d 2019/03/09 19:55  bike   
1  1297  2019/03/09 19:55 s/d 2019/03/09 19:54  food   
2  1394  2019/03/09 19:54 s/d 2019/03/09 18:56  shop   
3  1120  2019/03/09 18:56 s/d 2019/03/09 12:28  food   
4  2053  2019/03/09 12:28 s/d 2019/03/08 18:25   car   

                                         from_alamat   from_kelurahan  \
0                    gang ikhwan no.16,  sungai jawi      darat sekip   
1  neo shabu-shabu steak & shake, johar, jl. joha...  sungai bangkong   
2             alfamart pontianak mall, jl teuku umar      darat sekip   
3  parklife, jl. karimata no.64, sungai bangkong,...          mariana   
4        jl. tabrani ahmad no.12,  sungai jawi dalam         pal lima   

    from_kecamatan               from_latlng  \
0   pontianak kota     -0,0303277,109,297753   
1   pontianak kota       -0,02861,109,329253   
2   pontianak kota    -0,0301863,109,3356331

In [ ]:
# Ngatur Birthdate
# Customer Birthdate
if "customer_birthdate" in df.columns:
    df["customer_birthdate"] = pd.to_datetime(df["customer_birthdate"], errors="coerce")

    df["customer_age"] = df["start_time"].dt.year - df["customer_birthdate"].dt.year

    df.loc[
        (df["start_time"].dt.month < df["customer_birthdate"].dt.month) |
        ((df["start_time"].dt.month == df["customer_birthdate"].dt.month) &
         (df["start_time"].dt.day < df["customer_birthdate"].dt.day)),
        "customer_age"
    ] -= 1

# Driver Birthdate
if "driver_birthdate" in df.columns:
    df["driver_birthdate"] = pd.to_datetime(df["driver_birthdate"], errors="coerce")

    df["driver_age"] = df["start_time"].dt.year - df["driver_birthdate"].dt.year

    df.loc[
        (df["start_time"].dt.month < df["driver_birthdate"].dt.month) |
        ((df["start_time"].dt.month == df["driver_birthdate"].dt.month) &
         (df["start_time"].dt.day < df["driver_birthdate"].dt.day)),
        "driver_age"
    ] -= 1

# --- Contoh output ---
print("\nContoh data umur customer & driver:")
print(df[["id", "customer_birthdate", "customer_age", "driver_birthdate", "driver_age"]].head(10))


Contoh data umur customer & driver:
     id customer_birthdate  customer_age driver_birthdate  driver_age
0  1617         1994-02-05            25       1997-03-24          21
1  1297         2004-04-22            14       1976-07-26          42
2  1394         2000-01-07            19       1985-12-28          33
3  1120         1987-08-02            31       1993-06-10          25
4  2053         2004-01-23            15       1988-05-02          30
5  1574         1994-10-31            24       1992-07-12          26
6  1882         1988-04-12            30       1984-01-06          35
7  1958         1992-02-27            27       1991-12-21          27
8  2009         2004-04-01            14       1991-12-21          27
9  1671         1985-03-28            33       1995-01-12          24


In [ ]:
# Simpan hasil preprocessing ke file parquet
df.to_parquet("output_preprocessed.parquet", index=False)

print("\nPreprocessing selesai. Data tersimpan di file: output_preprocessed.parquet")


Preprocessing selesai. Data tersimpan di file: output_preprocessed.parquet


In [ ]:
# Simpan hasil preprocessing ke file CSV
df.to_csv("output_preprocessed.csv", index=False)

print("\nPreprocessing selesai. Data tersimpan di file: output_preprocessed.csv")


Preprocessing selesai. Data tersimpan di file: output_preprocessed.csv


In [ ]:
# MULAI ANALISIS DATA

In [ ]:
# 1. Baca hasil preprocessing
df = pd.read_parquet("output_preprocessed.parquet")

print("\nInfo Dataset untuk Analisis")
print("Jumlah baris:", len(df))
print("Jumlah kolom:", len(df.columns))
print("Kolom:\n", df.columns.tolist())


Info Dataset untuk Analisis
Jumlah baris: 1017
Jumlah kolom: 38
Kolom:
 ['id', 'date', 'mode', 'from_alamat', 'from_kelurahan', 'from_kecamatan', 'from_latlng', 'to_alamat', 'to_kelurahan', 'to_kecamatan', 'to_latlng', 'distance', 'amount_delivery', 'amount_merchant', 'transaction_amount_total', 'customer_id', 'customer_gender', 'customer_birthdate', 'driver_id', 'driver_gender', 'driver_birthdate', 'kendaraan_jenis', 'kendaraan_merk', 'merchant_id', 'merchant_name', 'merchant_category', 'customer_gender_encoded', 'kendaraan_jenis_encoded', 'start_time', 'end_time', 'duration_minutes', 'year', 'month', 'day', 'weekday', 'hour', 'customer_age', 'driver_age']


In [ ]:
# 1. Berapa banyak total transaksi?
print("Total transaksi:", len(df))

Total transaksi: 1017


In [ ]:
# 2. Berapa rata-rata total transaksi?
avg_transaction = df["transaction_amount_total"].mean()
print(f"Rata-rata transaksi: Rp {avg_transaction:,.2f}")

Rata-rata transaksi: Rp 36,206.98


In [ ]:
# 3. Bagaimana distribusi mode layanan?
mode_distribution = df["mode"].value_counts().to_frame("Jumlah")
mode_distribution["Persentase (%)"] = (df["mode"].value_counts(normalize=True) * 100).round(2)
print("Distribusi mode layanan:\n", mode_distribution)

Distribusi mode layanan:
       Jumlah  Persentase (%)
mode                        
bike     325           31.96
food     265           26.06
car      242           23.80
shop     185           18.19


In [ ]:
# 4. Bagaimana distribusi jenis kelamin driver berdasarkan order?
driver_gender_distribution = df["driver_gender"].value_counts().to_frame("Jumlah")
driver_gender_distribution["Persentase (%)"] = (df["driver_gender"].value_counts(normalize=True) * 100).round(2)
print("Distribusi jenis kelamin driver:\n", driver_gender_distribution)

Distribusi jenis kelamin driver:
                Jumlah  Persentase (%)
driver_gender                        
l                 728           71.58
p                 289           28.42


In [ ]:
# 5. Bagaimana distribusi jenis kelamin customer berdasarkan order?
customer_gender_distribution = df["customer_gender"].value_counts().to_frame("Jumlah")
customer_gender_distribution["Persentase (%)"] = (df["customer_gender"].value_counts(normalize=True) * 100).round(2)
print("Distribusi jenis kelamin customer:\n", customer_gender_distribution)

Distribusi jenis kelamin customer:
                  Jumlah  Persentase (%)
customer_gender                        
p                   529           52.02
l                   488           47.98


In [ ]:
# 6. Bagaimana distribusi jenis kendaraan?
kendaraan_distribution = df["kendaraan_jenis"].value_counts().to_frame("Jumlah")
kendaraan_distribution["Persentase (%)"] = (df["kendaraan_jenis"].value_counts(normalize=True) * 100).round(2)
print("Distribusi jenis kendaraan driver:\n", kendaraan_distribution)

Distribusi jenis kendaraan driver:
                  Jumlah  Persentase (%)
kendaraan_jenis                        
motor               775            76.2
mobil               242            23.8


In [ ]:
# 7. Apa merk kendaraan driver yang paling banyak muncul?
merk_distribution = df["kendaraan_merk"].value_counts().to_frame("Jumlah")
merk_distribution["Persentase (%)"] = (df["kendaraan_merk"].value_counts(normalize=True) * 100).round(2)
print("Distribusi merk kendaraan driver:\n", merk_distribution)

Distribusi merk kendaraan driver:
                 Jumlah  Persentase (%)
kendaraan_merk                        
bmw                216           21.24
honda              210           20.65
kawasaki           122           12.00
tvs                121           11.90
yamaha             117           11.50
suzuki              84            8.26
kia                 35            3.44
mitsubishi          33            3.24
toyota              28            2.75
audi                26            2.56
volvo               25            2.46


In [ ]:
# 8. Apa kategori merchant yang paling banyak dipilih? (exclude 'unknown')
merchant_filtered = df[df["merchant_name"].str.lower() != "unknown"]
merchant_category_distribution = merchant_filtered["merchant_category"].value_counts().to_frame("Jumlah")
merchant_category_distribution["Persentase (%)"] = (merchant_filtered["merchant_category"].value_counts(normalize=True) * 100).round(2)
print("Distribusi kategori merchant yang paling banyak dipilih:\n", merchant_category_distribution)

Distribusi kategori merchant yang paling banyak dipilih:
                    Jumlah  Persentase (%)
merchant_category                        
toko/swalayan         185           41.11
cafe                  113           25.11
warung makan           76           16.89
restaurant             42            9.33
jajanan                34            7.56


In [ ]:
# 9. Merchant mana yang menerima order terbanyak (exclude 'unknown')?
merchant_filtered = df[df["merchant_name"].str.lower() != "unknown"]
merchant_name_distribution = merchant_filtered["merchant_name"].value_counts().to_frame("Jumlah")
merchant_name_distribution["Persentase (%)"] = (merchant_filtered["merchant_name"].value_counts(normalize=True) * 100).round(2)
print("Distribusi merchant yang paling banyak dipilih:\n", merchant_name_distribution)

Distribusi merchant yang paling banyak dipilih:
                          Jumlah  Persentase (%)
merchant_name                                  
alfamart siantan hulu        27            6.00
indomaret teuku umar         26            5.78
indomaret kota baru          24            5.33
indomaret siantan hilir      24            5.33
alfamart pontianak mall      22            4.89
...                         ...             ...
glasshouse 55                 2            0.44
mie ayam ijo                  2            0.44
siobi pontianak               2            0.44
kings' kitchen and bar        1            0.22
pondok rasa                   1            0.22

[78 rows x 2 columns]


In [ ]:
# 10. Merchant mana yang menghasilkan omzet terbesar (exclude 'unknown')
merchant_filtered = df[df["merchant_name"].str.lower() != "unknown"]
merchant_omzet = merchant_filtered.groupby("merchant_name")["amount_merchant"].sum()
merchant_omzet_sorted = merchant_omzet.sort_values(ascending=False).round(2)
top_merchant = merchant_omzet_sorted.idxmax()
max_omzet = merchant_omzet_sorted.max()
print(f"Merchant dengan omzet terbesar: {top_merchant} (Rp {max_omzet:,.2f})")
print("Omzet per merchant (urut dari terbesar):\n", merchant_omzet_sorted)

Merchant dengan omzet terbesar: indomaret teuku umar (Rp 1,772,500.00)
Omzet per merchant (urut dari terbesar):
 merchant_name
indomaret teuku umar       1772500
alfamart siantan hulu      1677000
indomaret siantan hilir    1608500
indomaret kota baru        1504500
alfamart pontianak mall    1280500
                            ...   
crystal jade my bread       103000
glasshouse 55               100000
pondok rasa                  76000
siobi pontianak              60000
kings' kitchen and bar       31000
Name: amount_merchant, Length: 78, dtype: int64


In [ ]:
# 11.Bagaimana tren transaksi per tahun (year)?
transaksi_per_tahun = df.groupby("year")["transaction_amount_total"].sum().sort_index()
print("Total transaksi per tahun:\n", transaksi_per_tahun)

Total transaksi per tahun:
 year
2018    28612700
2019     8209800
Name: transaction_amount_total, dtype: int64


In [ ]:
# 12.Bagaimana tren transaksi per bulan (month)?
transaksi_per_bulan = df.groupby("month")["transaction_amount_total"].sum().reindex(range(1,13), fill_value=0)
print("Total transaksi per bulan:\n", transaksi_per_bulan)

Total transaksi per bulan:
 month
1      4194900
2      3093000
3       921900
4            0
5            0
6            0
7            0
8            0
9      3509800
10     4595900
11     3666700
12    16840300
Name: transaction_amount_total, dtype: int64


In [ ]:
# 13. Hari apa yang memiliki transaksi terbanyak?
transaksi_per_weekday = df.groupby("weekday")["transaction_amount_total"].sum()
transaksi_per_weekday = transaksi_per_weekday.sort_values(ascending=False)
print("Total transaksi per hari (weekday):\n", transaksi_per_weekday)

Total transaksi per hari (weekday):
 weekday
tuesday      6017800
monday       5944900
sunday       5322500
wednesday    5297900
thursday     5206400
saturday     4984500
friday       4048500
Name: transaction_amount_total, dtype: int64


In [ ]:
# 14. Pada jam berapa (hour) transaksi paling banyak terjadi?
transaksi_per_hour = df.groupby("hour")["transaction_amount_total"].sum()
transaksi_per_hour = transaksi_per_hour.sort_values(ascending=False)
print("Total transaksi per jam (hour):\n", transaksi_per_hour)

Total transaksi per jam (hour):
 hour
17    3128400
13    2708200
8     2638700
14    2429200
16    2405200
12    2359800
22    2238400
10    2122700
6     2098200
9     2096100
11    2091900
19    2068300
20    2037000
7     1975000
18    1723300
15    1509800
21    1192300
Name: transaction_amount_total, dtype: int64


In [ ]:
# 15. Apakah ada perbedaan jumlah order antara hari kerja vs akhir pekan?
df['day_type'] = df['weekday'].apply(lambda x: 'Akhir Pekan' if x in ['saturday', 'sunday'] else 'Hari Kerja')
order_count = df['day_type'].value_counts()
order_percent = (order_count / order_count.sum() * 100).round(2)
summary = pd.DataFrame({
    'Jumlah Order': order_count,
    'Persentase (%)': order_percent
})
print("Jumlah order Hari Kerja vs Akhir Pekan:\n", summary)

Jumlah order Hari Kerja vs Akhir Pekan:
              Jumlah Order  Persentase (%)
day_type                                 
Hari Kerja            730           71.78
Akhir Pekan           287           28.22


In [ ]:
# 16. Tren rata-rata distance per bulan
avg_distance_per_bulan = df.groupby("month")["distance"].mean().reindex(range(1,13), fill_value=0)
print("Rata-rata distance per bulan:\n", avg_distance_per_bulan.round(2))

Rata-rata distance per bulan:
 month
1     13.12
2      7.32
3      6.45
4      0.00
5      0.00
6      0.00
7      0.00
8      0.00
9      7.45
10    12.36
11     7.64
12    17.33
Name: distance, dtype: float64


In [ ]:
# 17. Apakah durasi lebih lama pada jam sibuk (pagi/sore)?
def categorize_rush_hour(hour):
    if 6 <= hour <= 9:
        return 'Pagi Sibuk'
    elif 16 <= hour <= 19:
        return 'Sore Sibuk'
    else:
        return 'Non Sibuk'
df['rush_hour'] = df['hour'].apply(categorize_rush_hour)
# Hitung rata-rata durasi dan jumlah order per kategori
summary = df.groupby('rush_hour')['duration_minutes'].agg(avg_duration_minutes='mean',total_orders='count').round(2)
def menit_ke_jammenit(menit):
    jam = int(menit // 60)
    sisa_menit = int(menit % 60)
    return f"{jam} jam {sisa_menit} menit"
summary['avg_duration_formatted'] = summary['avg_duration_minutes'].apply(menit_ke_jammenit)
print(summary.reset_index())

    rush_hour  avg_duration_minutes  total_orders avg_duration_formatted
0   Non Sibuk                263.26           531         4 jam 23 menit
1  Pagi Sibuk                205.81           252         3 jam 25 menit
2  Sore Sibuk                292.86           233         4 jam 52 menit


In [ ]:
# 18. Berapa rata-rata transaksi di pagi, siang, sore, malam?
def get_time_of_day(hour):
    if 5 <= hour < 12:
        return "Pagi"
    elif 12 <= hour < 15:
        return "Siang"
    elif 15 <= hour < 19:
        return "Sore"
    else:
        return "Malam"
df["time_of_day"] = df["hour"].apply(get_time_of_day)
# Hitung rata-rata transaksi kategori waktu
avg_amount_by_time = (df.groupby("time_of_day")["transaction_amount_total"].mean().sort_values(ascending=False))
print("Rata-rata transaction_amount_total per waktu (pagi, siang, sore, malam):\n", avg_amount_by_time)

Rata-rata transaction_amount_total per waktu (pagi, siang, sore, malam):
 time_of_day
Sore     37305.106383
Siang    36218.357488
Pagi     36173.888889
Malam    35051.162791
Name: transaction_amount_total, dtype: float64


In [ ]:
# 19. Apakah order lebih banyak dilakukan di awal bulan atau akhir bulan?
df['month_period'] = df['day'].apply(lambda x: 'Awal Bulan' if x <= 15 else 'Akhir Bulan')
order_count = df['month_period'].value_counts()
order_percent = (order_count / order_count.sum() * 100).round(2)
summary = pd.DataFrame({
    'Jumlah Order': order_count,
    'Persentase (%)': order_percent
})
print("Jumlah order Awal vs Akhir Bulan:\n", summary)

Jumlah order Awal vs Akhir Bulan:
               Jumlah Order  Persentase (%)
month_period                              
Akhir Bulan            641           63.03
Awal Bulan             376           36.97


In [ ]:
# 20. Berapa rata-rata jumlah transaksi per driver per bulan?
transactions_per_driver_month = df.groupby(['driver_id', 'month']).size().reset_index(name='num_transactions')
avg_transactions_per_driver = transactions_per_driver_month.groupby('driver_id')['num_transactions'].mean().round(2)
avg_transactions_sorted = avg_transactions_per_driver.sort_values(ascending=False)
print("Rata-rata jumlah transaksi per driver per bulan (dari paling banyak ke paling sedikit):")
print("Driver dengan transaksi terbanyak:")
print(avg_transactions_sorted.head(5))
print("Driver dengan transaksi tersedikit:")
print(avg_transactions_sorted.tail(5))

Rata-rata jumlah transaksi per driver per bulan (dari paling banyak ke paling sedikit):
Driver dengan transaksi terbanyak:
driver_id
88     7.33
94     6.33
91     5.83
78     5.83
105    5.17
Name: num_transactions, dtype: float64
Driver dengan transaksi tersedikit:
driver_id
89    3.67
75    3.43
80    3.33
76    3.29
92    3.14
Name: num_transactions, dtype: float64


In [ ]:
# 21.Kecamatan asal mana yang paling banyak order?
kecamatan_count = df['from_kecamatan'].value_counts()
kecamatan_percent = (kecamatan_count / kecamatan_count.sum() * 100).round(2)
summary = pd.DataFrame({
    'Jumlah Order': kecamatan_count,
    'Persentase (%)': kecamatan_percent
})
print("Jumlah order per kecamatan asal dengan persentase:\n", summary)

Jumlah order per kecamatan asal dengan persentase:
                     Jumlah Order  Persentase (%)
from_kecamatan                                  
pontianak kota               284           27.93
pontianak selatan            179           17.60
pontianak barat              164           16.13
pontianak utara              152           14.95
pontianak tenggara           151           14.85
pontianak timur               87            8.55


In [ ]:
# 22. Apakah jenis kendaraan memengaruhi rata-rata durasi perjalanan
avg_duration_per_vehicle = df.groupby('kendaraan_jenis')['duration_minutes'].mean().round(2)
avg_duration_per_vehicle_sorted = avg_duration_per_vehicle.sort_values(ascending=False)
# Konversi menit ke format jam:menit
def menit_ke_jammenit(menit):
    jam = int(menit // 60)
    sisa_menit = int(menit % 60)
    return f"{jam} jam {sisa_menit} menit"
avg_duration_formatted = avg_duration_per_vehicle_sorted.apply(menit_ke_jammenit)
# Gabungkan ke dataframe
summary = pd.DataFrame({
    'Rata-rata Durasi (menit)': avg_duration_per_vehicle_sorted,
    'Rata-rata Durasi (jam:menit)': avg_duration_formatted
})
# Tampilkan hasil
print("Rata-rata durasi perjalanan per jenis kendaraan:\n", summary)

Rata-rata durasi perjalanan per jenis kendaraan:
                  Rata-rata Durasi (menit) Rata-rata Durasi (jam:menit)
kendaraan_jenis                                                       
mobil                              262.29               4 jam 22 menit
motor                              253.77               4 jam 13 menit


In [ ]:
# 23. Apakah usia pelanggan memengaruhi pilihan kendaraan?
age_vehicle_count = df.groupby(['customer_age', 'kendaraan_jenis']).size().reset_index(name='num_orders')
age_total = age_vehicle_count.groupby('customer_age')['num_orders'].transform('sum')
age_vehicle_count['percent'] = (age_vehicle_count['num_orders'] / age_total * 100).round(2)
age_vehicle_count_sorted = age_vehicle_count.sort_values(['customer_age', 'percent'], ascending=[True, False])
print("Distribusi pilihan kendaraan per kelompok usia pelanggan:\n", age_vehicle_count_sorted)

Distribusi pilihan kendaraan per kelompok usia pelanggan:
     customer_age kendaraan_jenis  num_orders  percent
0             12           mobil           2   100.00
2             13           motor          24    66.67
1             13           mobil          12    33.33
4             14           motor          51    76.12
3             14           mobil          16    23.88
6             15           motor          32    76.19
5             15           mobil          10    23.81
8             16           motor          38    76.00
7             16           mobil          12    24.00
10            17           motor          57    79.17
9             17           mobil          15    20.83
12            18           motor          60    77.92
11            18           mobil          17    22.08
14            19           motor          30    73.17
13            19           mobil          11    26.83
16            20           motor          29    80.56
15            20       

In [ ]:
# 24. Kecamatan tujuan mana yang paling sering dituju?
to_kecamatan_count = df['to_kecamatan'].value_counts().reset_index()
to_kecamatan_count.columns = ['to_kecamatan', 'jumlah_order']
print("Kecamatan tujuan paling sering dituju:\n", to_kecamatan_count)

Kecamatan tujuan paling sering dituju:
          to_kecamatan  jumlah_order
0     pontianak timur           190
1     pontianak utara           180
2   pontianak selatan           179
3     pontianak barat           164
4      pontianak kota           161
5  pontianak tenggara           143


In [ ]:
# 25. Rata-rata distance dari masing-masing kecamatan asal?
avg_distance_kecamatan = (df.groupby("from_kecamatan")["distance"].mean().sort_values(ascending=False))
print("Rata-rata jarak tempuh per kacamatan asal:\n", avg_distance_kecamatan)

Rata-rata jarak tempuh per kacamatan asal:
 from_kecamatan
pontianak kota        24.865000
pontianak tenggara     8.385563
pontianak timur        8.132989
pontianak utara        7.783289
pontianak barat        7.446220
pontianak selatan      7.191006
Name: distance, dtype: float64


In [ ]:
# 26. Apakah ada perbedaan transaksi antar kecamatan asal?
avg_amount_by_kec = (df.groupby("from_kecamatan")["transaction_amount_total"].mean().sort_values(ascending=False))
print("Rata-rata total transaksi per kecamatan asal:\n", avg_amount_by_kec)

Rata-rata total transaksi per kecamatan asal:
 from_kecamatan
pontianak kota        48753.521127
pontianak selatan     42789.385475
pontianak barat       34482.926829
pontianak utara       32238.157895
pontianak tenggara    26472.847682
pontianak timur        8786.206897
Name: transaction_amount_total, dtype: float64


In [ ]:
# 27. Bagaimana pola rute yang paling sering terjadi?
df["route"] = df["from_kecamatan"] + " → " + df["to_kecamatan"]
route_counts = df["route"].value_counts().reset_index()
route_counts.columns = ["route", "count"]
print("10 Pola rute paling sering terjadi:\n", route_counts.head(10))

10 Pola rute paling sering terjadi:
                                     route  count
0        pontianak kota → pontianak barat     53
1         pontianak kota → pontianak kota     49
2      pontianak kota → pontianak selatan     48
3        pontianak kota → pontianak timur     47
4        pontianak kota → pontianak utara     46
5     pontianak kota → pontianak tenggara     41
6      pontianak selatan → pontianak kota     40
7       pontianak utara → pontianak timur     39
8  pontianak tenggara → pontianak selatan     37
9       pontianak barat → pontianak barat     35


In [ ]:
# 28. Bagaimana pola rute yang paling jarang terjadi?
df["route"] = df["from_kecamatan"] + " → " + df["to_kecamatan"]
route_counts = (df["route"].value_counts().reset_index())
route_counts.columns = ["route", "count"]
route_counts = route_counts.sort_values(by="count", ascending=True)
print("10 Pola rute paling jarang terjadi:\n", route_counts.head(10))

10 Pola rute paling jarang terjadi:
                                       route  count
35         pontianak timur → pontianak kota     10
34     pontianak timur → pontianak tenggara     12
33        pontianak timur → pontianak utara     14
32     pontianak tenggara → pontianak barat     15
31        pontianak timur → pontianak timur     16
28        pontianak utara → pontianak barat     17
29         pontianak barat → pontianak kota     17
30        pontianak timur → pontianak barat     17
27      pontianak timur → pontianak selatan     18
25  pontianak tenggara → pontianak tenggara     19


In [ ]:
# 29. Bagaimana pola rute yang paling sering terjadi (kelurahan)?
df["route"] = df["from_kelurahan"] + " → " + df["to_kelurahan"]
route_counts = df["route"].value_counts().reset_index()
route_counts.columns = ["route", "count"]
print("10 Pola rute paling sering terjadi:\n", route_counts.head(5))

10 Pola rute paling sering terjadi:
                             route  count
0    darat sekip → sungai beliung      8
1  bangka belitung darat → akcaya      7
2          mariana → siantan hulu      7
3            kota baru → pal lima      6
4      darat sekip → parit tokaya      6


In [ ]:
# 30. Bagaimana pola rute yang paling jarang terjadi (kelurahan)?
df["route"] = df["from_kelurahan"] + " → " + df["to_kelurahan"]
route_counts = (df["route"].value_counts().reset_index())
route_counts.columns = ["route", "count"]
route_counts = route_counts.sort_values(by="count", ascending=True)
print("10 Pola rute paling jarang terjadi:\n", route_counts.head(5))

10 Pola rute paling jarang terjadi:
                             route  count
527  sungai beliung → darat sekip      1
526     bansir laut → dalam bugis      1
525               tengah → saigon      1
524  tanjung hulu → tanjung hilir      1
523   bansir darat → bansir darat      1


In [ ]:
# 31. Apakah kecamatan asal berpengaruh terhadap durasi perjalanan?
df["duration_hours"] = df["duration_minutes"] / 60
avg_duration_by_kec = (df.groupby("from_kecamatan")["duration_hours"].mean().round(2).sort_values(ascending=False))
print("Rata-rata durasi perjalanan (jam) per kecamatan asal:", avg_duration_by_kec)

Rata-rata durasi perjalanan (jam) per kecamatan asal: from_kecamatan
pontianak timur       4.71
pontianak barat       4.67
pontianak tenggara    4.53
pontianak utara       4.26
pontianak kota        3.99
pontianak selatan     3.89
Name: duration_hours, dtype: float64


In [ ]:
# 31. Apakah kelurahan asal berpengaruh terhadap durasi perjalanan?
df["duration_hours"] = df["duration_minutes"] / 60
avg_duration_by_kel = (df.groupby("from_kelurahan")["duration_hours"].mean().round(2).sort_values(ascending=False))
print("Rata-rata durasi perjalanan (jam) per kelurahan asal paling lama:\n", avg_duration_by_kel.head(5))
# paling bentar
df["duration_hours"] = df["duration_minutes"] / 60
avg_duration_by_kel = (df.groupby("from_kelurahan")["duration_hours"].mean().round(2).sort_values(ascending=True))
print("\nRata-rata durasi perjalanan (jam) per kelurahan asal paling sebentar:\n", avg_duration_by_kel.head(5))

Rata-rata durasi perjalanan (jam) per kelurahan asal paling lama:
 from_kelurahan
bansir laut          5.98
dalam bugis          5.94
saigon               5.77
sungai jawi dalam    5.76
batu layang          5.74
Name: duration_hours, dtype: float64

Rata-rata durasi perjalanan (jam) per kelurahan asal paling sebentar:
 from_kelurahan
tambelan sampit    2.24
kota baru          2.55
siantan tengah     2.72
sungai bangkong    3.24
darat sekip        3.25
Name: duration_hours, dtype: float64


In [ ]:
# 32. Berapa rata-rata usia customer?
avg_customer_age = df["customer_age"].mean().round(2)
print(f"Rata-rata usia customer: {avg_customer_age} tahun")

Rata-rata usia customer: 24.44 tahun


In [ ]:
# 33. Berapa rata-rata usia customer berdasarkan mode dan jenis kendaraan?
avg_customer_age_grouped = (df.groupby(["mode", "kendaraan_jenis"])["customer_age"].mean().round(2) .reset_index())
print("Rata-rata usia customer per mode dan jenis kendaraan:\n", avg_customer_age_grouped)

Rata-rata usia customer per mode dan jenis kendaraan:
    mode kendaraan_jenis  customer_age
0  bike           motor         24.19
1   car           mobil         24.82
2  food           motor         23.76
3  shop           motor         25.37


In [ ]:
# 34. Berapa rata-rata usia driver?
avg_driver_age= df["driver_age"].mean().round(2)
print(f"Rata-rata usia driver: {avg_driver_age} tahun")

Rata-rata usia driver: 29.87 tahun


In [ ]:
# 33. Berapa rata-rata usia driver berdasarkan mode dan jenis kendaraan?
avg_driver_age_grouped = (df.groupby(["mode", "kendaraan_jenis"])["driver_age"].mean().round(2) .reset_index())
print("Rata-rata usia customer per mode dan jenis kendaraan:\n", avg_driver_age_grouped)

Rata-rata usia customer per mode dan jenis kendaraan:
    mode kendaraan_jenis  driver_age
0  bike           motor       29.70
1   car           mobil       30.29
2  food           motor       29.72
3  shop           motor       29.82


In [ ]:
# 34. Bagaiaman distribusi order berdasarkan gender per driver?
unique_driver_by_gender = (df.groupby("driver_gender")["driver_id"].nunique().reset_index().rename(columns={"driver_id": "jumlah_driver"}))
print("\nJumlah driver berdasarkan gender:\n", unique_driver_by_gender)


Jumlah driver berdasarkan gender:
   driver_gender  jumlah_driver
0             l             25
1             p             10


In [ ]:
# 35. Driver mana yang memiliki total pendapatan tertinggi dan terendah?
# Tertinggi
driver_income = (df.groupby("driver_id")["amount_delivery"].sum().reset_index().rename(columns={"amount_delivery": "total_income"}))
driver_income = driver_income.sort_values(by="total_income", ascending=False)
driver_income = driver_income.merge(df[["driver_id", "driver_gender"]].drop_duplicates(),on="driver_id",how="left")
print("5 Driver dengan pendapatan tertinggi:\n", driver_income.head(5))
# Terendah
driver_income = (df.groupby("driver_id")["amount_delivery"].sum().reset_index().rename(columns={"amount_delivery": "total_income"}))
driver_income = driver_income.sort_values(by="total_income", ascending=True)
driver_income = driver_income.merge(df[["driver_id", "driver_gender"]].drop_duplicates(),on="driver_id",how="left")
print("\n5 Driver dengan pendapatan terendah:\n", driver_income.head(5))

5 Driver dengan pendapatan tertinggi:
    driver_id  total_income driver_gender
0         96        519600             l
1         97        474000             l
2         99        472800             l
3         95        471600             l
4        109        451200             p

5 Driver dengan pendapatan terendah:
    driver_id  total_income driver_gender
0         80        107500             l
1         92        114000             l
2         76        131000             l
3         86        138000             l
4        106        145500             p


In [ ]:
# 36. Bagaimana distribusi total pendapatan berdasarkan gender?
income_by_gender = (df.groupby("driver_gender")["amount_delivery"].sum().reset_index().rename(columns={"amount_delivery": "total_pendapatan"}))
total_all_income = income_by_gender["total_pendapatan"].sum()
income_by_gender["persentase (%)"] = (
(income_by_gender["total_pendapatan"] / total_all_income * 100).round(2))
print("Distribusi total pendapatan berdasarkan gender driver:\n", income_by_gender)

Distribusi total pendapatan berdasarkan gender driver:
   driver_gender  total_pendapatan  persentase (%)
0             l           6089600           71.34
1             p           2446400           28.66


In [ ]:
# 37. Apakah pengemudi mobil menghasilkan pendapatan lebih besar dibanding motor?
income_by_vehicle = (df.groupby("kendaraan_jenis")["transaction_amount_total"].sum().reset_index().rename(columns={"transaction_amount_total": "total_pendapatan"}).sort_values(by="total_pendapatan", ascending=False))
total_all_income = income_by_vehicle["total_pendapatan"].sum()
income_by_vehicle["persentase (%)"] = ((income_by_vehicle["total_pendapatan"] / total_all_income * 100).round(2))
print("Total pendapatan berdasarkan jenis kendaraan:\n", income_by_vehicle)

Total pendapatan berdasarkan jenis kendaraan:
   kendaraan_jenis  total_pendapatan  persentase (%)
1           motor          33276500           90.37
0           mobil           3546000            9.63


In [ ]:
# 38. Korelasi antara usia driver dengan jumlah order yang diterima?
order_count_per_driver = (df.groupby(["driver_id", "driver_age"])["id"].count().reset_index().rename(columns={"id": "jumlah_order"}))
corr_value = order_count_per_driver["driver_age"].corr(order_count_per_driver["jumlah_order"])
print(f"Korelasi antara usia driver dengan jumlah order: {corr_value:.3f}")

Korelasi antara usia driver dengan jumlah order: 0.046


In [ ]:
# 39. Apakah driver laki-laki lebih sering membawa jarak jauh dibanding perempuan?
avg_distance_by_gender = df.groupby("driver_gender")["distance"].mean().round(2).reset_index(name="rata_rata_jarak")
print("Rata-rata jarak perjalanan berdasarkan gender driver:\n", avg_distance_by_gender.sort_values("rata_rata_jarak", ascending=False))

Rata-rata jarak perjalanan berdasarkan gender driver:
   driver_gender  rata_rata_jarak
1             p            15.09
0             l            11.49


In [ ]:
# 40. Apakah customer yang lebih muda cenderung order lebih sering?
order_count_per_customer = (df.groupby(["customer_id", "customer_age"])["id"].count().reset_index().rename(columns={"id": "jumlah_order"}))
corr_value = order_count_per_customer["customer_age"].corr(order_count_per_customer["jumlah_order"])
print(f"Korelasi usia customer dengan jumlah order: {corr_value:.3f}")

Korelasi usia customer dengan jumlah order: -0.060


In [ ]:
# 41. Customer usia berapa yang paling banyak dan paling sedikit melakukan transaksi?
# paling banyak
print ("Usia customer yang paling banyak melakukan transaksi:")
age_transaction_counts = (df.groupby("customer_age").size().reset_index(name="jumlah_transaksi").sort_values("jumlah_transaksi", ascending=False))
print(age_transaction_counts.head(5))
top = age_transaction_counts.iloc[0]
print(f"Customer usia {int(top['customer_age'])} tahun melakukan transaksi terbanyak: {top['jumlah_transaksi']} kali.")
# paling sedikit
print ("\nUsia customer yang paling sedikit melakukan transaksi:")
age_transaction_counts = (df.groupby("customer_age").size().reset_index(name="jumlah_transaksi").sort_values("jumlah_transaksi", ascending=True))
print(age_transaction_counts.head(0))
top = age_transaction_counts.iloc[0]
print(f"Customer usia {int(top['customer_age'])} tahun melakukan transaksi tersedikit: {top['jumlah_transaksi']} kali.")

Usia customer yang paling banyak melakukan transaksi:
    customer_age  jumlah_transaksi
6             18                77
19            31                75
5             17                72
2             14                67
21            33                59
Customer usia 18 tahun melakukan transaksi terbanyak: 77 kali.

Usia customer yang paling sedikit melakukan transaksi:
Empty DataFrame
Columns: [customer_age, jumlah_transaksi]
Index: []
Customer usia 40 tahun melakukan transaksi tersedikit: 1 kali.


In [ ]:
# 42. Bagaimana distribusi total pendapatan per mode layanan (diluar biaya merchant)?
total_all_income = df["amount_delivery"].sum()
income_by_mode = (df.groupby("mode")["amount_delivery"].sum().reset_index().rename(columns={"amount_delivery": "total_pendapatan"}).sort_values(by="total_pendapatan", ascending=False))
income_by_mode["persentase (%)"] = ((income_by_mode["total_pendapatan"] / total_all_income * 100).round(2))
print("Distribusi total pendapatan berdasarkan mode layanan:\n",income_by_mode.to_string(index=False))

Distribusi total pendapatan berdasarkan mode layanan:
 mode  total_pendapatan  persentase (%)
 car           3546000           41.54
food           2185000           25.60
shop           1554000           18.21
bike           1251000           14.66


In [ ]:
# 43. Berapa rata-rata total pemasukkan driver per order?
avg_transaction = df["amount_delivery"].mean()
print(f"Rata-rata transaksi: Rp {avg_transaction:,.2f}")

Rata-rata transaksi: Rp 8,393.31


In [ ]:
# 44. Berapa rata-rata total pengeluaran customer untuk membeli barang per order?
avg_transaction = df.loc[df["amount_merchant"] != 0, "amount_merchant"].mean()
print(f"Rata-rata transaksi barang/makanan: Rp {avg_transaction:,.2f}")

Rata-rata transaksi barang/makanan: Rp 62,858.89


In [ ]:
# 45. Berapa rata-rata total pemasukkan driver per order berdasarkan jenis kendaraan?
avg_income_by_vehicle = (df.groupby("kendaraan_jenis")["amount_delivery"].mean().round(2).reset_index().rename(columns={"amount_delivery": "rata_rata_pemasukan"}))
print("Rata-rata pemasukan driver per order berdasarkan jenis kendaraan:\n",avg_income_by_vehicle)

# Bagaimana rata - rata pendapatan jika hanya layanan antar?
df_bc = df[df["mode"].isin(["bike", "car"])]
avg_delivery_fee = df_bc.groupby("mode")["amount_delivery"].mean().round(2).reset_index()
avg_delivery_fee.rename(columns={"amount_delivery": "avg_delivery_fee"}, inplace=False)
print("\nRata-rata biaya antar per mode layanan (Bike / Car):\n",avg_delivery_fee)

Rata-rata pemasukan driver per order berdasarkan jenis kendaraan:
   kendaraan_jenis  rata_rata_pemasukan
0           mobil             14652.89
1           motor              6438.71

Rata-rata biaya antar per mode layanan (Bike / Car):
    mode  amount_delivery
0  bike          3849.23
1   car         14652.89


In [ ]:
# 46. Yang mana jarak untuk bike dan car paling jauh dan dekat?
# Paling jauh
df_bc = df[df["mode"].isin(["bike", "car"])]
longest_trip_by_mode = (df_bc.loc[df_bc["distance"] == df_bc.groupby("mode")["distance"].transform("max")][["mode", "from_kecamatan", "to_kecamatan", "distance", "amount_delivery"]].sort_values(by="distance", ascending=False))
print("Perjalanan paling jauh per mode layanan (Bike / Car):\n",longest_trip_by_mode)
# Paling dekat
shortest_trip_by_mode = (df_bc.loc[df_bc["distance"] == df_bc.groupby("mode")["distance"].transform("min")][["mode", "from_kecamatan", "to_kecamatan", "distance", "amount_delivery"]].sort_values(by="distance", ascending=True))
print("\nPerjalanan paling dekat per mode layanan (Bike / Car):\n",shortest_trip_by_mode)

Perjalanan paling jauh per mode layanan (Bike / Car):
       mode      from_kecamatan        to_kecamatan  distance  amount_delivery
1003  bike     pontianak utara  pontianak tenggara     17.92             9500
364    car  pontianak tenggara     pontianak timur     15.90            32400

Perjalanan paling dekat per mode layanan (Bike / Car):
      mode   from_kecamatan     to_kecamatan  distance  amount_delivery
578  bike   pontianak kota  pontianak utara      2.09              500
426   car  pontianak timur  pontianak timur      2.51             3600
